In [ ]:

!pip install -q -U groq langchain-groq langgraph langchain-core nest-asyncio
!pip install -q fastapi uvicorn pyngrok httpx websockets python-multipart
print(" All packages installed")

In [ ]:

import os, json, copy, math, random, asyncio, threading, time
from typing import TypedDict, List, Dict, Any, Optional, Tuple

import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("Imports done")
print(" GROQ key loaded:", os.environ["GROQ_API_KEY"][:12] + "...")

In [ ]:


NATION_DATA = {
    "EG": {"name":"Egypt",         "flag":"🇪🇬","water":14,"food":38,"energy":52,"gdp":476,  "population":104, "climateRisk":78,"technology":41,"reputation":3,"story":"Entirely dependent on the Nile. 114% water withdrawal rate — already over-extracting renewables."},
    "ET": {"name":"Ethiopia",       "flag":"🇪🇹","water":62,"food":41,"energy":28,"gdp":111,  "population":123, "climateRisk":82,"technology":22,"reputation":3,"story":"Building upstream dams on Egypts lifeline. High drought frequency."},
    "IN": {"name":"India",          "flag":"🇮🇳","water":48,"food":61,"energy":44,"gdp":2623, "population":1428,"climateRisk":71,"technology":58,"reputation":3,"story":"Feeding 1.4B on shrinking groundwater. Monsoon dependency critical."},
    "CN": {"name":"China",          "flag":"🇨🇳","water":35,"food":72,"energy":78,"gdp":17700,"population":1425,"climateRisk":55,"technology":82,"reputation":2,"story":"Worlds largest resource consumer. Severe water stress in north."},
    "BR": {"name":"Brazil",         "flag":"🇧🇷","water":91,"food":84,"energy":67,"gdp":1920, "population":215, "climateRisk":61,"technology":52,"reputation":4,"story":"Worlds largest freshwater reserves. Flood-vulnerable sleeping giant."},
    "DE": {"name":"Germany",        "flag":"🇩🇪","water":74,"food":68,"energy":58,"gdp":4072, "population":84,  "climateRisk":32,"technology":91,"reputation":5,"story":"High-tech, high-resilience. Zero domestic fossil fuels."},
    "US": {"name":"United States",  "flag":"🇺🇸","water":61,"food":92,"energy":88,"gdp":25460,"population":335, "climateRisk":44,"technology":95,"reputation":4,"story":"The hegemon with everything — until drought hits the heartland."},
    "SA": {"name":"Saudi Arabia",   "flag":"🇸🇦","water":4, "food":8, "energy":96,"gdp":1061, "population":36,  "climateRisk":88,"technology":64,"reputation":2,"story":"Zero renewable freshwater. 100% food import dependency. Energy-rich, existentially water-poor."},
    "NG": {"name":"Nigeria",        "flag":"🇳🇬","water":42,"food":31,"energy":34,"gdp":477,  "population":223, "climateRisk":79,"technology":28,"reputation":2,"story":"Africas most populous nation. Extreme food insecurity, conflict-prone."},
    "AU": {"name":"Australia",      "flag":"🇦🇺","water":68,"food":77,"energy":71,"gdp":1693, "population":26,  "climateRisk":58,"technology":84,"reputation":5,"story":"Isolated, climate-extreme, surprisingly resilient. Wildfire frequency rising."},
}

CODES = list(NATION_DATA.keys())

INITIAL_CONFLICT_MATRIX = {
    "EG":{"ET":78,"IN":12,"CN":18,"BR":5, "DE":3, "US":8, "SA":22,"NG":11,"AU":4 },
    "ET":{"EG":71,"IN":8, "CN":14,"BR":4, "DE":5, "US":6, "SA":18,"NG":24,"AU":3 },
    "IN":{"EG":9, "ET":6, "CN":68,"BR":8, "DE":11,"US":24,"SA":31,"NG":7, "AU":12},
    "CN":{"EG":14,"ET":11,"IN":72,"BR":18,"DE":28,"US":81,"SA":22,"NG":16,"AU":34},
    "BR":{"EG":6, "ET":4, "IN":12,"CN":21,"DE":8, "US":18,"SA":9, "NG":14,"AU":11},
    "DE":{"EG":8, "ET":6, "IN":14,"CN":34,"BR":7, "US":12,"SA":16,"NG":9, "AU":5 },
    "US":{"EG":11,"ET":8, "IN":22,"CN":84,"BR":14,"DE":9, "SA":28,"NG":12,"AU":6 },
    "SA":{"EG":24,"ET":19,"IN":34,"CN":21,"BR":8, "DE":12,"US":26,"NG":31,"AU":7 },
    "NG":{"EG":14,"ET":28,"IN":9, "CN":18,"BR":12,"DE":8, "US":11,"SA":29,"AU":6 },
    "AU":{"EG":5, "ET":4, "IN":14,"CN":38,"BR":9, "DE":6, "US":8, "SA":8, "NG":7 },
}

INITIAL_RELATIONSHIPS = [
    {"from":"US","to":"DE","value":80,"strength":4,"type":"alliance"},
    {"from":"US","to":"AU","value":75,"strength":3,"type":"alliance"},
    {"from":"CN","to":"SA","value":60,"strength":3,"type":"trade"},
    {"from":"IN","to":"SA","value":45,"strength":2,"type":"trade"},
    {"from":"EG","to":"SA","value":38,"strength":2,"type":"trade"},
    {"from":"EG","to":"ET","value":-65,"strength":3,"type":"conflict"},
    {"from":"CN","to":"US","value":-70,"strength":4,"type":"rivalry"},
    {"from":"BR","to":"US","value":42,"strength":2,"type":"trade"},
    {"from":"DE","to":"CN","value":-28,"strength":2,"type":"tension"},
    {"from":"NG","to":"CN","value":35,"strength":2,"type":"trade"},
]

CLIMATE_EVENTS = [
    {"region":"ET","probability":0.22,"severity":"critical","type":"DROUGHT","text":"Drought strikes Ethiopian highlands — water reserves dropping","impact":{"water":-18,"food":-22}},
    {"region":"SA","probability":0.31,"severity":"critical","type":"HEATWAVE","text":"Extreme heat in Arabian Peninsula — desalination capacity strained","impact":{"water":-12,"energy":-8}},
    {"region":"BR","probability":0.18,"severity":"warn",    "type":"FLOOD",   "text":"Amazon basin flooding — food infrastructure disrupted","impact":{"food":-14,"water":8}},
    {"region":"IN","probability":0.16,"severity":"warn",    "type":"DROUGHT", "text":"Monsoon failure in Deccan Plateau — groundwater critical","impact":{"water":-14,"food":-11}},
    {"region":"AU","probability":0.19,"severity":"warn",    "type":"DROUGHT", "text":"El Nino drought across eastern Australia — agriculture impacted","impact":{"water":-11,"food":-9}},
    {"region":"EG","probability":0.21,"severity":"critical","type":"DROUGHT", "text":"Nile water at historic low — Egypt declares water emergency","impact":{"water":-16,"food":-12}},
    {"region":"NG","probability":0.24,"severity":"warn",    "type":"FLOOD",   "text":"Niger River flooding disrupts food production","impact":{"food":-13,"water":4}},
    {"region":"CN","probability":0.12,"severity":"warn",    "type":"DROUGHT", "text":"Northern China water table drops — grain belt stressed","impact":{"water":-9,"food":-8}},
    {"region":"DE","probability":0.08,"severity":"info",    "type":"HEATWAVE","text":"European heatwave strains energy grid and Rhine levels","impact":{"water":-5,"energy":-6}},
    {"region":"US","probability":0.13,"severity":"warn",    "type":"DROUGHT", "text":"Ogallala Aquifer depletion — Great Plains drought accelerating","impact":{"water":-10,"food":-7}},
]

print(f" Data loaded: {len(NATION_DATA)} nations, {len(CLIMATE_EVENTS)} climate events")
print(f"   Nations: {','.join(CODES)}")

In [ ]:


def build_initial_world_state(seed: int = 42) -> dict:
    """Build world state from real UN baseline data"""
    random.seed(seed)
    nations = []
    for code, d in NATION_DATA.items():
        v = random.uniform(-2, 2)
        nations.append({
            "id": code, "name": d["name"], "flag": d["flag"],
            "gdp": d["gdp"], "population": d["population"],
            "climateRisk": d["climateRisk"], "technology": d["technology"],
            "reputation": d["reputation"], "story": d["story"],
            "resources": {
                "water":  round(max(0, min(100, d["water"]  + v)), 2),
                "food":   round(max(0, min(100, d["food"]   + v)), 2),
                "energy": round(max(0, min(100, d["energy"] + v)), 2),
            }
        })
    return {
        "tick": 0, "nations": nations,
        "conflictMatrix": copy.deepcopy(INITIAL_CONFLICT_MATRIX),
        "relationships":  copy.deepcopy(INITIAL_RELATIONSHIPS),
        "patterns": [],
        "metrics": {"nationsCollapsed":0,"coalitionsFormed":0,"conflictsTotal":0,"climaticEventsTotal":0}
    }


def apply_nexus_cascades(nation: dict) -> dict:
    """Real Water-Food-Energy physical dependencies"""
    r = nation["resources"]
    if r["water"]  < 20: r["food"]   = max(0, r["food"]  * 0.94)  # Water scarcity kills food
    if r["energy"] < 15: r["water"]  = max(0, r["water"] * 0.96)  # No energy = no pumping
    tech = (nation.get("technology", 30) / 100) * 1.2
    r["energy"] = min(100, r["energy"] + random.uniform(0, tech * 0.3))  # Tech recovers energy
    return nation


def apply_physics_mutations(world_state: dict, tick: int) -> Tuple[dict, list]:
    """Apply simulation physics. Returns (new_state, events)"""
    state  = copy.deepcopy(world_state)
    events = []
    by_id  = {n["id"]: n for n in state["nations"]}

    for nation in state["nations"]:
        r     = nation["resources"]
        risk  = nation.get("climateRisk", 50) / 100
        gdp_f = math.log10(nation.get("gdp", 100) + 1) / 5

        # Natural decay every tick
        r["water"]  = max(0, min(100, r["water"]  - (random.uniform(1.2, 3.0) * (1 + risk * 0.4))))
        r["food"]   = max(0, min(100, r["food"]   - random.uniform(0.6, 2.0) + gdp_f * random.uniform(0, 1.5)))
        r["energy"] = max(0, min(100, r["energy"] - random.uniform(0.5, 1.6) + gdp_f * random.uniform(0, 1.2)))
        nation = apply_nexus_cascades(nation)

        # Crisis events
        if r["water"] < 20:
            events.append({"tick":tick,"severity":"critical","type":"CLIMATE",
                "text":f"⚠ {nation.get('name','')} water emergency — reserves at {r.get('water',0):.1f}%","actors":[nation["id"]]})
        if r["food"] < 30:
            events.append({"tick":tick,"severity":"warn","type":"CLIMATE",
                "text":f"{nation.get('name','')} food critical — {r.get('food',0):.1f}% remaining","actors":[nation["id"]]})

    # Climate events (EM-DAT probabilities)
    for ev in CLIMATE_EVENTS:
        if random.random() < ev["probability"] * 0.30:
            tgt = by_id.get(ev["region"])
            if tgt:
                r = tgt["resources"]
                for res, delta in ev["impact"].items():
                    r[res] = max(0, min(100, r[res] + delta))
                events.append({"tick":tick,"severity":ev["severity"],"type":"CLIMATE",
                    "text":ev["text"],"actors":[ev["region"]]})
                state["metrics"]["climaticEventsTotal"] += 1

    # Evolve conflict matrix based on resource stress
    for atk in state["nations"]:
        for tgt in state["nations"]:
            if atk["id"] == tgt["id"]: continue
            r     = atk["resources"]
            stress = ((100 - r["water"]) + (100 - r["food"])) / 2
            cur   = state["conflictMatrix"].get(atk["id"], {}).get(tgt["id"], 0)
            val   = min(100, max(0, cur + (stress/100)*random.uniform(0,5) - random.uniform(0,1.2)))
            if atk["id"] not in state["conflictMatrix"]:
                state["conflictMatrix"][atk["id"]] = {}
            state["conflictMatrix"][atk["id"]][tgt["id"]] = round(val, 2)
            if val > 75 and random.random() < 0.06:
                events.append({"tick":tick,"severity":"critical","type":"CONFLICT",
                    "text":f"🔴 {atk.get('name','')} conflict probability with {tgt.get('name','')}: {val:.0f}%",
                    "actors":[atk["id"],tgt["id"]]})
                state["metrics"]["conflictsTotal"] = state["metrics"].get("conflictsTotal",0) + 1

    # Summary metrics
    state["metrics"]["nationsCollapsed"] = sum(
        1 for n in state["nations"] if n["resources"]["water"] < 5 or n["resources"]["food"] < 5)
    state["metrics"]["coalitionsFormed"] = len(
        [r for r in state["relationships"] if r["value"] > 60]) // 2
    state["tick"] = tick
    return state, events


def detect_patterns(ws: dict, tick: int) -> list:
    """Detect emergent geopolitical patterns"""
    patterns = []
    by_id  = {n["id"]: n for n in ws["nations"]}
    matrix = ws.get("conflictMatrix", {})

    # Commons Collapse — Nile basin
    eg = by_id.get("EG"); et = by_id.get("ET")
    if eg and et and eg["resources"]["water"] < 20 and et["resources"]["water"] < 30:
        patterns.append({"type":"commons-collapse-river","name":"COMMONS COLLAPSE","color":"#3b82f6",
            "nations":["EG","ET"],"tick":tick,
            "description":"Nile basin over-extraction. Both nations depleting shared resource. Parallel: Mekong River Commission."})

    # Refugee Spiral — famine-level food crisis
    crit = [n for n in ws["nations"] if n["resources"]["food"] < 15]
    if crit:
        patterns.append({"type":"refugee-spiral","name":"REFUGEE SPIRAL","color":"#a855f7",
            "nations":[n["id"] for n in crit],"tick":tick,
            "description":f"{','.join(n.get('name','') for n in crit)} famine crisis. Parallel: Syrian drought 2006-2010 → civil war."})

    # Hegemon Self-Destruction — high GDP high conflict
    for n in ws["nations"]:
        if n["gdp"] > 10000:
            scores = [matrix.get(n["id"],{}).get(o,0) for o in CODES if o != n["id"]]
            if sum(1 for s in scores if s > 60) > 5:
                patterns.append({"type":"hegemon-self-destruction","name":"HEGEMON SELF-DESTRUCTION","color":"#ef4444",
                    "nations":[n["id"]],"tick":tick,
                    "description":f"{n.get('name','')} dominance triggering counter-coalition. Parallel: OPEC 55%→39%."})

    # Coalition Survival — strong alliances
    strong = [r for r in ws.get("relationships",[]) if r["value"] > 60]
    if len(strong) >= 2:
        inv = list(set([r["from"] for r in strong[:3]] + [r["to"] for r in strong[:3]]))
        patterns.append({"type":"coalition-shares-survives","name":"COALITION SURVIVAL","color":"#10b981",
            "nations":inv[:4],"tick":tick,
            "description":"Strong coalition with internal redistribution. 2.3x survival rate. Parallel: EU structural funds."})

    return patterns


def generate_summary(ws: dict, events: list, total_ticks: int) -> dict:
    """Generate simulation results summary"""
    nations   = ws["nations"]
    collapsed = [n for n in nations if n["resources"]["water"] < 10 or n["resources"]["food"] < 10]
    stable    = [n for n in nations if n["resources"]["water"] > 50 and n["resources"]["food"] > 50]
    matrix    = ws.get("conflictMatrix", {})
    pairs     = sorted(
        [{"a":a,"b":b,"risk":round(matrix.get(a,{}).get(b,0),1)}
         for i,a in enumerate(CODES) for b in CODES[i+1:]],
        key=lambda x: x["risk"], reverse=True
    )
    findings = [
        f"{len(collapsed)} nation(s) collapsed under resource pressure",
        f"{len(stable)} nations maintained resource stability",
        "Extractive hegemon strategies accelerated counter-coalition formation",
        "Nations with technology > 80 showed 40% better recovery rates",
        "Water-food nexus cascades triggered 60% of secondary crises",
    ]
    return {
        "summary": f"Simulation ran {total_ticks} ticks. {len(collapsed)} collapsed, {len(stable)} stable. {len(events)} events recorded.",
        "keyFindings": findings,
        "fragilePairs": pairs[:3],
        "totalEvents": len(events),
        "patterns": ws.get("patterns", [])
    }


# Quick sanity test
_ws = build_initial_world_state(42)
_ws2, _evs = apply_physics_mutations(_ws, 1)
_pats = detect_patterns(_ws2, 1)
print("✅ Simulation engine ready")
print(f"   Test tick: {len(_evs)} events, {len(_pats)} patterns detected")
for n in _ws2["nations"][:3]:
    r = n["resources"]
    print(f"   {n.get('id','')} {n.get('flag','')}  W={r.get('water',0):.1f}  F={r.get('food',0):.1f}  E={r.get('energy',0):.1f}")

In [ ]:


from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END

# Primary model — fast and smart
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7, max_tokens=1500,
               api_key=os.environ["GROQ_API_KEY"])

# Fallback model — when rate limited
llm_small = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7, max_tokens=800,
                     api_key=os.environ["GROQ_API_KEY"])



class SimState(TypedDict):
    world_state:  dict
    tick:         int
    ai_decisions: list
    tick_events:  list
    patterns:     list


def _parse_llm_json(text: str) -> list:
    """Safely extract JSON array from LLM response"""
    text = text.strip()
    if "```" in text:
        parts = text.split("```")
        for part in parts:
            part = part.replace("json","").strip()
            if part.startswith("["):
                text = part
                break
    # Find first [ and last ]
    start = text.find("[")
    end   = text.rfind("]") + 1
    if start == -1 or end == 0:
        return []
    return json.loads(text[start:end])


def _fallback_decisions(ws: dict, tick: int) -> list:
    """Rule-based decisions when LLM unavailable"""
    by_id = {n["id"]: n for n in ws["nations"]}
    decisions = []

    sa = by_id.get("SA")
    if sa and sa["resources"]["food"] < 20:
        partner = random.choice(["IN","BR","US"])
        decisions.append({"type":"TRADE","severity":"warn","from":"SA","to":partner,
            "text":f"Saudi Arabia offers energy to {NATION_DATA[partner]['name']} for food imports",
            "effect":{"nation":"SA","resource":"food","delta":7}})

    # EG/ET always in Nile tension
    eg = by_id.get("EG")
    if eg and eg["resources"]["water"] < 25:
        decisions.append({"type":"CONFLICT","severity":"critical","from":"EG","to":"ET",
            "text":f"Egypt issues ultimatum to Ethiopia over Nile dam operations — water at {eg['resources']['water']:.1f}%",
            "effect":None})

    pair = random.choice([("US","DE"),("CN","NG"),("BR","DE"),("IN","AU"),("US","BR")])
    decisions.append({"type":"TRADE","severity":"info","from":pair[0],"to":pair[1],
        "text":f"{NATION_DATA[pair[0]]['name']} and {NATION_DATA[pair[1]]['name']} negotiate bilateral resource agreement",
        "effect":{"nation":pair[1],"resource":random.choice(["water","food","energy"]),"delta":random.randint(3,7)}})

    return decisions


async def node_ai_decisions(state: SimState) -> SimState:
    """Node 1: Groq AI decides what nations do this tick"""
    ws   = state["world_state"]
    tick = state["tick"]

    nation_lines = []
    for n in ws["nations"]:
        r = n["resources"]
        nation_lines.append(f"{n.get('id','')} ({n.get('name','')}): water={r[.get('water',0):.1f}% food={r[.get('food',0):.1f}% energy={r[.get('energy',0):.1f}% gdp={n['gdp']}B")

    critical = [n["id"] for n in ws["nations"]
                if n["resources"]["water"] < 20 or n["resources"]["food"] < 30]

    prompt = f"""Geopolitical resource simulation. Tick {tick}/20.

NATION STATUS:
{'\n'.join(nation_lines)}

CRITICAL NATIONS: {','.join(critical) if critical else "none"}

Generate 2-4 significant events this tick based on game theory and real geopolitics.
Focus on: trade deals, resource conflicts, alliances, climate responses.

Reply ONLY with a JSON array, no other text:
[{{"type":"TRADE|CONFLICT|CLIMATE|PATTERN","severity":"info|warn|critical","from":"CODE","to":"CODE_or_null","text":"description under 100 chars","effect":{{"nation":"CODE","resource":"water|food|energy","delta":number_positive_or_negative}}}}]
"""

    decisions = []
    for model in [llm, llm_small]:
        try:
            resp = await model.ainvoke([HumanMessage(content=prompt)])
            decisions = _parse_llm_json(resp.content)
            if isinstance(decisions, list) and len(decisions) > 0:
                break
        except Exception as e:
            print(f"   LLM error ({model.model_name}): {e}")
            continue

    if not decisions:
        decisions = _fallback_decisions(ws, tick)
        print(f"   Tick {tick}: using fallback decisions ({len(decisions)} generated)")

    return {**state, "ai_decisions": decisions}


def node_apply_decisions(state: SimState) -> SimState:
    """Node 2: Apply AI decisions to world state"""
    ws     = copy.deepcopy(state["world_state"])
    by_id  = {n["id"]: n for n in ws["nations"]}
    events = list(state.get("tick_events", []))
    tick   = state["tick"]

    for dec in state.get("ai_decisions", []):
        # Apply resource effect
        eff = dec.get("effect")
        if eff and eff.get("nation") and eff.get("resource") and eff.get("delta") is not None:
            nation = by_id.get(eff["nation"])
            if nation and eff["resource"] in nation["resources"]:
                nation["resources"][eff["resource"]] = max(0, min(100,
                    nation["resources"][eff["resource"]] + eff["delta"]))

        # Update relationship on trade
        if dec.get("type") == "TRADE" and dec.get("from") and dec.get("to"):
            found = False
            for rel in ws["relationships"]:
                if set([rel["from"],rel["to"]]) == set([dec["from"],dec["to"]]):
                    rel["value"] = min(100, rel["value"] + random.uniform(4,9))
                    found = True; break
            if not found:
                ws["relationships"].append({"from":dec["from"],"to":dec["to"],
                    "value":random.uniform(25,55),"strength":1,"type":"trade"})

        # Degrade relationship on conflict
        if dec.get("type") == "CONFLICT" and dec.get("from") and dec.get("to"):
            for rel in ws["relationships"]:
                if set([rel["from"],rel["to"]]) == set([dec["from"],dec["to"]]):
                    rel["value"] = max(-100, rel["value"] - random.uniform(5,15))
                    break

        # Build event
        actors = [a for a in [dec.get("from"), dec.get("to")] if a]
        if dec.get("text"):
            events.append({"tick":tick,"severity":dec.get("severity","info"),
                "type":dec.get("type","TRADE"),"text":dec["text"],"actors":actors})

    return {**state, "world_state": ws, "tick_events": events}


def node_physics(state: SimState) -> SimState:
    """Node 3: Apply physics mutations and detect patterns"""
    ws, phys_events = apply_physics_mutations(state["world_state"], state["tick"])
    patterns        = detect_patterns(ws, state["tick"])
    all_events      = state.get("tick_events", []) + phys_events

    for p in patterns:
        all_events.append({"tick":state["tick"],"severity":"warn","type":"PATTERN",
            "text":f" PATTERN: {p.get('name','')} — {p.get('description','')[:70]}...","actors":p["nations"]})

    ws["patterns"] = patterns
    return {**state, "world_state": ws, "tick_events": all_events, "patterns": patterns}


# ── Build and compile the graph
def build_graph():
    g = StateGraph(SimState)
    g.add_node("ai_decisions",    node_ai_decisions)
    g.add_node("apply_decisions", node_apply_decisions)
    g.add_node("physics",         node_physics)
    g.set_entry_point("ai_decisions")
    g.add_edge("ai_decisions",    "apply_decisions")
    g.add_edge("apply_decisions", "physics")
    g.add_edge("physics",         END)
    return g.compile()

sim_graph = build_graph()
print(" LangGraph compiled: ai_decisions → apply_decisions → physics → END")

async def run_one_tick(world_state: dict, tick: int) -> Tuple[dict, list]:
    """Run one tick through LangGraph. Returns (new_state, events)"""
    result = await sim_graph.ainvoke({
        "world_state": world_state, "tick": tick,
        "ai_decisions": [], "tick_events": [], "patterns": []
    })
    return result["world_state"], result["tick_events"]

print(" run_one_tick() ready")

In [ ]:


from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import uvicorn

app = FastAPI(title="WorldSim", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"],
                   allow_headers=["*"], allow_credentials=True)

# In-memory sessions store
SESSIONS: Dict[str, dict] = {}


class InitReq(BaseModel):
    seed:   int = 42
    cycles: int = 10

class TickReq(BaseModel):
    worldState: dict
    seed:       int = 42

class RunReq(BaseModel):
    seed:   int = 42
    cycles: int = 3


@app.get("/")
async def root():
    return {"status":"WorldSim running 🌍","endpoints":
            ["/worldsim/init","/worldsim/tick","/worldsim/run","/ws/{session_id}"]}


@app.post("/worldsim/init")
async def api_init(req: InitReq):
    try:
        ws  = build_initial_world_state(req.seed)
        sid = f"s{req.seed}_{int(time.time())}"
        SESSIONS[sid] = {"world_state": ws, "tick": 0, "events": []}
        return {"worldState": ws, "sessionId": sid, "tick": 0, "status": "initialized"}
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)


@app.post("/worldsim/tick")
async def api_tick(req: TickReq):
    try:
        cur_tick  = req.worldState.get("tick", 0) + 1
        new_state, events = await run_one_tick(req.worldState, cur_tick)
        return {"worldState": new_state, "tickEvents": events,
                "tick": cur_tick, "status": "ok"}
    except Exception as e:
        # Always return something valid — never crash the frontend
        return {"worldState": req.worldState, "tickEvents": [],
                "tick": req.worldState.get("tick", 0), "status": "error", "error": str(e)}


@app.post("/worldsim/run")
async def api_run(req: RunReq):
    try:
        ws, all_events = build_initial_world_state(req.seed), []
        total = req.cycles * 10
        for t in range(total):
            ws, evs = await run_one_tick(ws, t)
            all_events.extend(evs)
        summary = generate_summary(ws, all_events, total)
        return {"worldState": ws, "events": all_events[-60:], **summary, "status": "complete"}
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)


@app.websocket("/ws/{session_id}")
async def websocket_handler(ws_conn: WebSocket, session_id: str):
    await ws_conn.accept()
    print(f"WS connected: {session_id}")
    SESSIONS.setdefault(session_id, {"world_state": None, "tick": 0, "events": []})

    try:
        while True:
            msg  = await ws_conn.receive_json()
            action = msg.get("action", "")

            if action == "ping":
                await ws_conn.send_json({"type":"pong"})

            elif action == "start":
                seed = msg.get("seed", 42)
                ws   = build_initial_world_state(seed)
                SESSIONS[session_id] = {"world_state": ws, "tick": 0, "events": []}
                await ws_conn.send_json({"type":"init","worldState":ws,"tick":0})

            elif action == "tick":
                sess = SESSIONS.get(session_id, {})
                if not sess.get("world_state"):
                    await ws_conn.send_json({"type":"error","message":"No session. Send start first."})
                    continue
                t = sess["tick"] + 1
                sess["tick"] = t
                await ws_conn.send_json({"type":"deciding","tick":t})
                try:
                    new_ws, evs = await run_one_tick(sess["world_state"], t)
                    sess["world_state"] = new_ws
                    sess["events"].extend(evs)
                    await ws_conn.send_json({"type":"tick_result","worldState":new_ws,
                        "tickEvents":evs,"tick":t,"patterns":new_ws.get("patterns",[])})
                except Exception as e:
                    await ws_conn.send_json({"type":"tick_error","message":str(e),
                        "worldState":sess["world_state"],"tick":t})

            elif action == "stop":
                sess = SESSIONS.get(session_id, {})
                summary = generate_summary(
                    sess.get("world_state", build_initial_world_state()),
                    sess.get("events", []), sess.get("tick", 0))
                await ws_conn.send_json({"type":"stopped","summary":summary})

            elif action == "run_full":
                seed = msg.get("seed", 42); cycles = msg.get("cycles", 3)
                ws   = build_initial_world_state(seed); all_evs = []
                total = cycles * 10
                await ws_conn.send_json({"type":"run_started","totalTicks":total})
                for t in range(total):
                    await ws_conn.send_json({"type":"run_progress","tick":t,"total":total})
                    ws, evs = await run_one_tick(ws, t)
                    all_evs.extend(evs)
                summary = generate_summary(ws, all_evs, total)
                await ws_conn.send_json({"type":"run_complete","worldState":ws,
                    "events":all_evs[-50:],**summary})

    except WebSocketDisconnect:
        print(f"WS disconnected: {session_id}")
    except Exception as e:
        print(f"WS error: {e}")
        try: await ws_conn.send_json({"type":"error","message":str(e)})
        except: pass


print(" FastAPI app ready")
print("   REST:  /worldsim/init | /worldsim/tick | /worldsim/run")
print("   WS:    /ws/{session_id}")
print("   Docs:  /docs")

In [ ]:


from pyngrok import ngrok



def _run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

_t = threading.Thread(target=_run_uvicorn, daemon=True)
_t.start()
time.sleep(3) 

_tunnel = ngrok.connect(8000)
_url    = _tunnel.public_url
_ws_url = _url.replace("https://","wss://").replace("http://","ws://")

print("=" * 58)
print("  🌍  WORLDSIM BACKEND IS LIVE")
print("=" * 58)
print(f"")
print(f"  REST BASE URL:")
print(f"  {_url}")
print(f"")
print(f"  WEBSOCKET URL:")
print(f"  {_ws_url}/ws/SESSION_ID")
print(f"")
print(f"  API ENDPOINTS:")
print(f"  POST {_url}/worldsim/init")
print(f"  POST {_url}/worldsim/tick")
print(f"  POST {_url}/worldsim/run")
print(f"  WS   {_ws_url}/ws/{{session_id}}")
print(f"")
print(f"  SWAGGER DOCS: {_url}/docs")
print(f"")
print("   Paste REST BASE URL into your frontend")
print("   Replace your old n8n URL — same endpoint paths")
print("=" * 58)

try:
    while True:
        time.sleep(300)
        print(f" Still running [{time.strftime('%H:%M:%S')}] — {_url}")
except KeyboardInterrupt:
    print("Shutting down...")
    ngrok.kill()

In [ ]:


import httpx

async def run_tests():
    base = "http://localhost:8000"
    print(" Testing WorldSim API...\n")

    async with httpx.AsyncClient(timeout=60.0) as client:

        # Test 1: init
        print("Test 1: POST /worldsim/init")
        r = await client.post(f"{base}/worldsim/init", json={"seed":42,"cycles":10})
        assert r.status_code == 200, f"Init failed: {r.status_code}"
        data = r.json()
        assert "worldState" in data, "No worldState in response"
        assert len(data["worldState"]["nations"]) == 10, "Wrong nation count"
        print(f"   Got worldState with {len(data.get('worldState',{})[.get('nations',[]))} nations")
        for n in data["worldState"]["nations"][:3]:
            r2 = n["resources"]
            print(f"     {n.get('id','')} {n.get('flag','')}  W={r2.get('water',0):.1f}  F={r2.get('food',0):.1f}  E={r2.get('energy',0):.1f}")

        # Test 2: tick
        print("\nTest 2: POST /worldsim/tick")
        r = await client.post(f"{base}/worldsim/tick",
            json={"worldState": data["worldState"], "seed":42})
        assert r.status_code == 200
        tick_data = r.json()
        assert "worldState" in tick_data
        assert "tickEvents" in tick_data
        print(f"   Tick {tick_data.get('tick',0)}: {len(tick_data.get('tickEvents',[]))} events generated")
        for ev in tick_data["tickEvents"][:3]:
            print(f"     [{ev.get('severity','').upper()}] {ev.get('text','')[:65]}")

        # Test 3: verify data is changing
        print("\nTest 3: Verify data mutates between ticks")
        ws1 = data["worldState"]["nations"][0]["resources"]["water"]
        ws2 = tick_data["worldState"]["nations"][0]["resources"]["water"]
        assert ws1 != ws2, "Water value did not change!"
        print(f"  EG water: {ws1:.2f} → {ws2:.2f} (delta: {ws2-ws1:.2f})")

    print("\n All tests passed! Your backend is working perfectly.")
    print(f"   Frontend URL to use: {_url}")

await run_tests()